# Round 10: target encoding and config diversity on the counts+stochastic winner - S6E10

Round 9's Exp 29 (fold-local value counts inside the stochastic 3-seed scheme) became the new
preferred candidate: mean OOF 0.95933, all five fold deltas positive vs the Exp 22 bar. Counts are
now the most productive feature-engineering direction in the campaign. Two bounded tests built on
the Exp 29 winner:

- **Exp 31: cross-fitted smoothed target encoding.** Counts tell the model how common a value is;
  a target encoding tells it how satisfied that value's passengers were - the label-carrying
  cousin of the count features. Leakage-safe construction, strict on both label boundaries:
  within each outer fold the 10% early-stopping split happens BEFORE any encoding is computed;
  fit rows get encodings cross-fitted over 5 inner folds (a row's own label never enters its
  encoding), early-stop rows get encodings from fit-rows-only tables (no early-stop label touches
  any early-stop encoding), and validation rows get full outer-train tables (early-stop labels
  included - legitimate: those rows are training data relative to the outer validation fold).
  Smoothing m=100 shrinks rare values toward the global mean; inner folds are label-blind
  (plain shuffled KFold) so fold composition cannot carry label information either.
- **Exp 32: config diversity under the same averaging scheme.** Exp 29 averages one config over 3
  seeds. Averaging {num_leaves 63, 127} x 3 seeds (6 models) tests whether mild capacity diversity
  decorrelates the fold errors further than seeds alone.

Bar and reading rules: the bar is now Exp 29's round-9 fold constants, paired per-fold deltas,
2x-spread band as a decision heuristic (never proof). The incumbent is still recomputed in-run as
the determinism reference. The refit guard fires only on a working-signal win over the Exp 29 bar;
highest measured OOF vs preferred refit candidate stays explicit. The 0.960 target is a mean-OOF
target - a single fold above it is not attainment.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E22_FOLDS_R6 = [0.95886, 0.95756, 0.95953, 0.95875, 0.95922]        # stochastic averaging, round 6
E29_FOLDS_R9 = [0.95938, 0.95804, 0.96033, 0.95933, 0.95956]        # counts + stochastic, the bar
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100  # smoothing strength for target encodings

In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings for the rows of X themselves: a row's own label never enters its
    # encoding - not through the value table and not through the smoothing mean (the global
    # mean is recomputed per inner fold from itr rows only). Inner folds are label-blind
    # (plain shuffled KFold) so fold composition carries no label information either.
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    # encodings for rows of X_new from tables built on (X_ref, y_ref) only
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False, target_encode=False, **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        # early-stop split happens BEFORE any target encoding, so no early-stop label can
        # touch any early-stop encoding (round-10 pre-dispatch audit finding, fixed)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        if target_encode:
            te_fit = te_crossfit(X_fit, y_fit, COUNT_COLS)          # own-label excluded per row
            te_es = te_apply(X_fit, y_fit, X_es, COUNT_COLS)        # fit-rows-only tables
            te_va = te_apply(X_tr, y_tr, X_va, COUNT_COLS)          # full outer-train tables (legit)
            X_fit = pd.concat([X_fit.reset_index(drop=True), te_fit.reset_index(drop=True)], axis=1)
            X_es = pd.concat([X_es.reset_index(drop=True), te_es.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features; TE columns: {len(COUNT_COLS)}')

Full train: 699,635 rows, 21 raw features; TE columns: 21


## Reference: incumbent recompute

Same-run incumbent with the determinism check, as every round.

In [3]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc

Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 1: AUC 0.95851 (best_iteration=296)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 2: AUC 0.95751 (best_iteration=265)


...

  INC fold 3: AUC 0.95907 (best_iteration=212)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 4: AUC 0.95841 (best_iteration=183)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 5: AUC 0.95885 (best_iteration=555)
  determinism vs round-3 constants: max abs diff 0.00000 (PASS - identical folds)


## Exp 31: counts + cross-fitted smoothed target encoding

The Exp 29 scheme plus 21 target-encoded columns (one per raw column), leakage-safe on both
boundaries: the early-stopping split is taken before encoding, fit rows are cross-fitted over
label-blind inner folds, early-stop rows are encoded from fit-rows-only tables, and validation
rows from full outer-train tables. Smoothing m=100. Stochastic config, seeds 42/43/44, OOF
averaged per row.

In [4]:
print('Exp 31: counts + cross-fitted smoothed TE + stochastic, seeds 42/43/44, averaged')
oof_seeds31 = {}
iters31 = None
for seed in [42, 43, 44]:
    o, a, it = oof_lgbm(features, target, f'E31-s{seed}', seed=seed,
                        count_features=True, target_encode=True, **STOCH)
    oof_seeds31[seed] = o
    if seed == 42:
        iters31 = it  # kept for the refit best-iteration basis
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof31 = np.mean([oof_seeds31[s] for s in [42, 43, 44]], axis=0)
aucs31 = fold_aucs(oof31, target)
print(f'Exp 31 (averaged) fold AUCs: {[round(a, 5) for a in aucs31]}')
print(f'Exp 31 mean OOF AUC: {np.mean(aucs31):.5f} +/- {np.std(aucs31):.5f} | pooled: {roc_auc_score(target, oof31):.6f}')

Exp 31: counts + cross-fitted smoothed TE + stochastic, seeds 42/43/44, averaged


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s42 fold 1: AUC 0.95989 (best_iteration=284)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s42 fold 2: AUC 0.95864 (best_iteration=271)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s44 fold 4: AUC 0.95995 (best_iteration=381)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E31-s44 fold 5: AUC 0.95976 (best_iteration=258)
  seed 44 mean AUC: 0.95966
Exp 31 (averaged) fold AUCs: [0.95998, 0.95873, 0.96056, 0.96, 0.96]


Exp 31 mean OOF AUC: 0.95986 +/- 0.00060 | pooled: 0.959854


## Exp 32: config diversity under counts + stochastic averaging

Same feature set as Exp 29 (counts only), but the average spans {num_leaves 63, 127} x seeds
42/43/44 - 6 models. Tests whether mild capacity diversity adds anything on top of seed diversity
within this scheme; bounded to these two configs.

In [5]:
print('Exp 32: counts + stochastic, 3 seeds x {63, 127} leaves, 6-model average')
LEAF_CONFIGS = [63, 127]
oof_runs32, iters32 = {}, {}
for leaves in LEAF_CONFIGS:
    for seed in [42, 43, 44]:
        o, a, it = oof_lgbm(features, target, f'E32-l{leaves}-s{seed}', seed=seed,
                            count_features=True, num_leaves=leaves, **STOCH)
        oof_runs32[(leaves, seed)] = o
        iters32[(leaves, seed)] = it
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof32 = np.mean(list(oof_runs32.values()), axis=0)
aucs32 = fold_aucs(oof32, target)
print(f'Exp 32 (averaged) fold AUCs: {[round(a, 5) for a in aucs32]}')
print(f'Exp 32 mean OOF AUC: {np.mean(aucs32):.5f} +/- {np.std(aucs32):.5f} | pooled: {roc_auc_score(target, oof32):.6f}')

Exp 32: counts + stochastic, 3 seeds x {63, 127} leaves, 6-model average


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E32-l63-s42 fold 1: AUC 0.95911 (best_iteration=285)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E32-l63-s42 fold 2: AUC 0.95800 (best_iteration=266)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E32-l127-s44 fold 4: AUC 0.95907 (best_iteration=187)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E32-l127-s44 fold 5: AUC 0.95941 (best_iteration=287)
  leaves=127 seed 44 mean AUC: 0.95927
Exp 32 (averaged) fold AUCs: [0.95952, 0.95833, 0.96041, 0.9594, 0.95968]


Exp 32 mean OOF AUC: 0.95947 +/- 0.00067 | pooled: 0.959464


## Results, verdicts, and refit guard

Each candidate pairs against the in-run incumbent, the Exp 22 folds, and the Exp 29 folds (the
bar). Refit fires only on a WORKING SIGNAL better vs the Exp 29 bar; on a tie or loss, Exp 29 and
submission_round9.csv stand. Null results are bounded to the tested variant - no detected
increment in this test, not a closed family.

In [6]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label

print('Exp 31: counts + target encoding + stochastic')
m, s, lab = paired_verdict(aucs31, INC_FOLDS)
print(f'  vs incumbent (in-run):        mean delta {m:+.5f} (spread {s:.5f}) - {lab}')
m, s, lab = paired_verdict(aucs31, E22_FOLDS_R6)
print(f'  vs Exp 22 folds (round 6):    mean delta {m:+.5f} (spread {s:.5f}) - {lab}')
m31, s31, lab31 = paired_verdict(aucs31, E29_FOLDS_R9)
print(f'  vs Exp 29 folds (round 9 bar): mean delta {m31:+.5f} (spread {s31:.5f}) - {lab31}')

print('Exp 32: counts + stochastic config diversity')
m, s, lab = paired_verdict(aucs32, INC_FOLDS)
print(f'  vs incumbent (in-run):        mean delta {m:+.5f} (spread {s:.5f}) - {lab}')
m, s, lab = paired_verdict(aucs32, E22_FOLDS_R6)
print(f'  vs Exp 22 folds (round 6):    mean delta {m:+.5f} (spread {s:.5f}) - {lab}')
m32, s32, lab32 = paired_verdict(aucs32, E29_FOLDS_R9)
print(f'  vs Exp 29 folds (round 9 bar): mean delta {m32:+.5f} (spread {s32:.5f}) - {lab32}')

verdicts = {'31': (m31, lab31, float(np.mean(aucs31))), '32': (m32, lab32, float(np.mean(aucs32)))}
winners = [k for k, (m, lab, _) in verdicts.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda k: verdicts[k][2]) if winners else None
print()
if win:
    print(f'REFIT GUARD: Exp {win} clears the Exp 29 bar under the paired band -> refit fires (exploratory, descriptive signal).')
else:
    print('REFIT GUARD: nothing cleared the Exp 29 bar under the paired band. Exp 29 remains the '
          'preferred candidate; submission_round9.csv is untouched.')

Exp 31: counts + target encoding + stochastic
  vs incumbent (in-run):        mean delta +0.00139 (spread 0.00017) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 22 folds (round 6):    mean delta +0.00107 (spread 0.00016) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 29 folds (round 9 bar): mean delta +0.00053 (spread 0.00017) - WORKING SIGNAL: better (descriptive, not proof)
Exp 32: counts + stochastic config diversity
  vs incumbent (in-run):        mean delta +0.00100 (spread 0.00019) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 22 folds (round 6):    mean delta +0.00068 (spread 0.00014) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 29 folds (round 9 bar): mean delta +0.00014 (spread 0.00008) - NO SIGNAL (within 2x per-fold delta spread - no action)

REFIT GUARD: Exp 31 clears the Exp 29 bar under the paired band -> refit fires (exploratory, descriptive signal).


In [7]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    preds = []
    if win == '31':
        basis = int(round(np.mean(iters31)))
        tr_c, (te_c,) = add_counts(features, [test_features])
        te_full = te_crossfit(tr_c, target, COUNT_COLS)          # full-train cross-fitted
        te_test = te_apply(features, target, test_features, COUNT_COLS)
        X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
        X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
        X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
        for seed in [42, 43, 44]:
            cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=basis,
                       n_jobs=-1, random_state=seed, verbose=-1)
            cfg.update(STOCH)
            model = lgb.LGBMClassifier(**cfg)
            model.fit(X_full, target)
            preds.append(model.predict_proba(X_test)[:, 1])
        scheme = 'Exp 31: counts + cross-fitted TE + stochastic, 3 seeds'
    else:  # win == '32'
        basis = int(round(np.mean([np.mean(iters32[(l, 42)]) for l in LEAF_CONFIGS])))
        tr_c, (te_c,) = add_counts(features, [test_features])
        X_full, X_test = to_lgbm(tr_c), to_lgbm(te_c)
        for leaves in LEAF_CONFIGS:
            for seed in [42, 43, 44]:
                cfg = dict(learning_rate=0.05, num_leaves=leaves, n_estimators=basis,
                           n_jobs=-1, random_state=seed, verbose=-1)
                cfg.update(STOCH)
                model = lgb.LGBMClassifier(**cfg)
                model.fit(X_full, target)
                preds.append(model.predict_proba(X_test)[:, 1])
        scheme = 'Exp 32: counts + stochastic, 3 seeds x {63,127} leaves'
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': np.mean(preds, axis=0)})
    sub.to_csv('submission_round10.csv', index=False)
    print(f'submission_round10.csv written: {len(sub):,} rows | {scheme} | n_estimators basis {basis} '
          f'(in-run seed-42 fold-mean best_iteration). Local file only - NOT submitted; Andrea submits.')

submission_round10.csv written: 299,844 rows | Exp 31: counts + cross-fitted TE + stochastic, 3 seeds | n_estimators basis 275 (in-run seed-42 fold-mean best_iteration). Local file only - NOT submitted; Andrea submits.


In [8]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round10_oof.npz',
         incumbent=oof_inc, exp31=oof31, exp32=oof32, target=target.to_numpy())
meta = {
    'round': 10,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configs': {
        'incumbent': {'note': 'Exp 8 config, in-run reference'},
        'exp31': {'scheme': 'counts + cross-fitted smoothed TE (m=100, inner 5-fold) + STOCH, seeds 42/43/44 averaged'},
        'exp32': {'scheme': 'counts + STOCH, num_leaves {63,127} x seeds 42/43/44, 6-model average'},
        'stoch': STOCH,
    },
    'fold_aucs': {'incumbent': INC_FOLDS, 'exp31': aucs31, 'exp32': aucs32},
}
(cache_dir / 'round10_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round10_oof.npz + round10_oof.json')

cached oof_cache/round10_oof.npz + round10_oof.json


## What happens next

- If Exp 31 beats the bar: value-level label information adds something on top of counts in this
  test - the refit fires and follow-ups tune the smoothing strength or restrict TE to the columns
  where it earned its place.
- If Exp 31 shows no detected increment: bounded to this encoding (m=100, value-level, these
  columns) - smoothing strength and column subset stay open, but the cheap version is tested.
- If Exp 32 beats the bar: capacity diversity decorrelates errors further than seeds alone -
  follow-ups widen the config mix (learning rate, min_child_samples) one dimension at a time.
- If both are null: the Exp 29 winner stands, and the next direction is the pending external-data
  question (host-linked original dataset, needs Andrea's explicit OK first) or count-based crosses.